# Data Exploration & Quality Review

In [ ]:
import json, sys, random
from pathlib import Path
from collections import Counter
import pandas as pd

# Add project root
sys.path.insert(0, str(Path('.').resolve().parent))
PROJECT_ROOT = Path('.').resolve().parent
DATA_DIR = PROJECT_ROOT / 'data'

print(f'Project root: {PROJECT_ROOT}')
print(f'Data dir:     {DATA_DIR}')
print(f'Data dir exists: {DATA_DIR.exists()}')

## 1. Extraction Summary

In [ ]:
try:
    extracted_dir = DATA_DIR / 'extracted'
    if not extracted_dir.exists():
        print('No data/extracted/ directory found. Run extraction scripts first.')
    else:
        rows = []
        subdir_counts = Counter()

        for json_file in sorted(extracted_dir.rglob('*.json')):
            rel = json_file.relative_to(extracted_dir)
            subdir = rel.parts[0] if len(rel.parts) > 1 else '(root)'
            subdir_counts[subdir] += 1

            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            source_id = data.get('source_id', json_file.stem)
            sections = data.get('sections', [])
            total_chars = sum(len(s.get('text', '')) for s in sections)

            rows.append({
                'subdirectory': subdir,
                'source_id': source_id,
                'num_sections': len(sections),
                'total_characters': total_chars,
            })

        if not rows:
            print('Extracted directory exists but contains no JSON files.')
        else:
            print('=== Files per subdirectory ===')
            for subdir, count in sorted(subdir_counts.items()):
                print(f'  {subdir}: {count} file(s)')
            print()

            df_extracted = pd.DataFrame(rows)
            print(f'Total extracted files: {len(df_extracted)}')
            print(f'Total sections: {df_extracted["num_sections"].sum()}')
            print(f'Total characters: {df_extracted["total_characters"].sum():,}')
            print()
            display(df_extracted)

except Exception as e:
    print(f'Error loading extracted data: {e}')

## 2. Section Analysis

In [ ]:
try:
    extracted_dir = DATA_DIR / 'extracted'
    if not extracted_dir.exists():
        print('No data/extracted/ directory found. Run extraction scripts first.')
    else:
        all_sections = []
        for json_file in extracted_dir.rglob('*.json'):
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            for section in data.get('sections', []):
                all_sections.append({
                    'source': data.get('source_id', json_file.stem),
                    'title': section.get('title', '(untitled)'),
                    'char_count': len(section.get('text', '')),
                    'text': section.get('text', ''),
                })

        if not all_sections:
            print('No sections found in extracted files.')
        else:
            df_sections = pd.DataFrame(all_sections)
            print(f'Total sections: {len(df_sections)}')
            print()

            # Length distribution via binned value_counts
            print('=== Section text length distribution (character counts) ===')
            bins = [0, 100, 500, 1000, 2000, 5000, 10000, 50000, float('inf')]
            labels = ['0-100', '101-500', '501-1k', '1k-2k', '2k-5k', '5k-10k', '10k-50k', '50k+']
            df_sections['length_bin'] = pd.cut(df_sections['char_count'], bins=bins, labels=labels)
            print(df_sections['length_bin'].value_counts().sort_index().to_string())
            print()

            # Shortest and longest
            shortest = df_sections.loc[df_sections['char_count'].idxmin()]
            longest = df_sections.loc[df_sections['char_count'].idxmax()]
            print(f'Shortest section: "{shortest["title"]}" ({shortest["char_count"]} chars)')
            print(f'Longest section:  "{longest["title"]}" ({longest["char_count"]} chars)')
            print()

            # 3 random samples
            print('=== 3 Random Section Samples ===')
            sample_n = min(3, len(df_sections))
            samples = df_sections.sample(n=sample_n, random_state=42)
            for i, (_, row) in enumerate(samples.iterrows(), 1):
                preview = row['text'][:500] + ('...' if len(row['text']) > 500 else '')
                print(f'\n--- Sample {i}: "{row["title"]}" ({row["char_count"]} chars) ---')
                print(preview)

except Exception as e:
    print(f'Error analyzing sections: {e}')

## 3. Chunk Analysis

In [ ]:
try:
    chunks_file = DATA_DIR / 'chunks' / 'all_chunks.jsonl'
    if not chunks_file.exists():
        print('No data/chunks/all_chunks.jsonl found. Run chunking scripts first.')
    else:
        chunks = []
        with open(chunks_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    chunks.append(json.loads(line))

        if not chunks:
            print('Chunks file is empty.')
        else:
            df_chunks = pd.DataFrame(chunks)
            print(f'Total chunks: {len(df_chunks)}')

            if 'token_count' in df_chunks.columns:
                total_tokens = df_chunks['token_count'].sum()
                print(f'Total tokens: {total_tokens:,}')
                print()
                print('=== Token count distribution ===')
                print(df_chunks['token_count'].describe().to_string())
                print(f'Median: {df_chunks["token_count"].median()}')
            else:
                print('No token_count field found in chunks.')

            print()
            if 'source_id' in df_chunks.columns:
                print('=== Chunks per source_id ===')
                print(df_chunks['source_id'].value_counts().to_string())
                print()

            if 'content_type' in df_chunks.columns:
                print('=== Content type breakdown ===')
                print(df_chunks['content_type'].value_counts().to_string())
                print()

            print('=== DataFrame Summary ===')
            display(df_chunks.describe(include='all'))

except Exception as e:
    print(f'Error loading chunks: {e}')

## 4. Generated Dataset Samples

In [ ]:
try:
    generated_dir = DATA_DIR / 'generated'
    dataset_types = ['qa_pairs', 'conversations', 'completions', 'classification']

    if not generated_dir.exists():
        print('No data/generated/ directory found. Run generation scripts first.')
    else:
        for dtype in dataset_types:
            print(f'\n{"=" * 60}')
            print(f'Dataset type: {dtype}')
            print('=' * 60)

            type_dir = generated_dir / dtype
            if not type_dir.exists():
                print('No data generated yet - run generation scripts first')
                continue

            # Collect all examples from JSONL files in this directory
            all_examples = []
            for jfile in type_dir.glob('*.jsonl'):
                with open(jfile, 'r', encoding='utf-8') as f:
                    for line in f:
                        line = line.strip()
                        if line:
                            all_examples.append(json.loads(line))

            # Also check for JSON files
            for jfile in type_dir.glob('*.json'):
                with open(jfile, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                if isinstance(data, list):
                    all_examples.extend(data)
                else:
                    all_examples.append(data)

            if not all_examples:
                print('No data generated yet - run generation scripts first')
                continue

            print(f'Total examples: {len(all_examples)}')
            sample_n = min(2, len(all_examples))
            samples = random.sample(all_examples, sample_n)
            for i, sample in enumerate(samples, 1):
                print(f'\n--- Sample {i} ---')
                print(json.dumps(sample, indent=2, ensure_ascii=False)[:2000])

except Exception as e:
    print(f'Error loading generated data: {e}')

## 5. Validated Dataset Summary

In [ ]:
try:
    validated_dir = DATA_DIR / 'validated'
    if not validated_dir.exists():
        print('No data/validated/ directory found.')
        print('After generating datasets, run validation scripts to populate this directory.')
    else:
        jsonl_files = list(validated_dir.glob('*.jsonl'))
        if not jsonl_files:
            print('No validated JSONL files found.')
            print('After generating datasets, run validation scripts to populate this directory.')
        else:
            total = 0
            rows = []
            for jfile in sorted(jsonl_files):
                count = 0
                with open(jfile, 'r', encoding='utf-8') as f:
                    for line in f:
                        if line.strip():
                            count += 1
                total += count
                rows.append({'file': jfile.name, 'count': count})

            df_validated = pd.DataFrame(rows)
            print(f'Total validated examples: {total}')
            print()
            display(df_validated)

except Exception as e:
    print(f'Error loading validated data: {e}')

## 6. Training Data Overview

In [ ]:
try:
    training_dir = DATA_DIR / 'training'
    splits = ['train.jsonl', 'val.jsonl', 'test.jsonl']

    if not training_dir.exists():
        print('No data/training/ directory found.')
        print('Run the full pipeline (extract -> chunk -> generate -> validate -> split) first.')
    else:
        found_any = False
        for split_name in splits:
            split_file = training_dir / split_name
            if not split_file.exists():
                continue
            found_any = True

            examples = []
            with open(split_file, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if line:
                        examples.append(json.loads(line))

            df_split = pd.DataFrame(examples)
            print(f'\n{"=" * 50}')
            print(f'Split: {split_name}  ({len(df_split)} rows)')
            print('=' * 50)

            if 'format' in df_split.columns:
                print('\nFormat distribution:')
                print(df_split['format'].value_counts().to_string())

            if 'source_id' in df_split.columns:
                print('\nSource distribution:')
                print(df_split['source_id'].value_counts().to_string())
            elif 'source' in df_split.columns:
                print('\nSource distribution:')
                print(df_split['source'].value_counts().to_string())

        if not found_any:
            print('No train/val/test.jsonl files found in data/training/.')
            print('Run the full pipeline (extract -> chunk -> generate -> validate -> split) first.')

except Exception as e:
    print(f'Error loading training data: {e}')